# RQ3 Assumption Checks: Score Spread and Formula Checks

This notebook checks the basic assumptions behind the Pearson and
Spearman correlations used in `experiment/run_rq3_experiment.py`, and
visualises the relationship between the score components.

For each profile (run), the experiment computes **unrounded** per-song
values:

- `final_score` — the score used for ranking,
- `cosine_similarity` — similarity between the song and the ideal vector,
- `avoid_penalty` — proportion of singing time on avoid notes,
- `favorite_overlap` — proportion of singing time on favourite notes.

Pearson's *r* is used for:

- final_score vs cosine_similarity,
- final_score vs avoid_penalty,

and Spearman's ρ is used for:

- cosine_similarity vs favorite_overlap.

Here we:

- load the actual per-song values from `experiment_results/RQ3_results.json`,
- for a couple of example runs, draw scatterplots for the three pairs,
- and briefly discuss what these plots tell us about linearity/monotonicity
  and the choice of Pearson vs Spearman.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# This notebook lives in experiment_results/assumption_checks
EXP_DIR = Path("..")
RQ3_PATH = EXP_DIR / "RQ3_results.json"

with RQ3_PATH.open(encoding="utf-8") as f:
    rq3 = json.load(f)

per_run = rq3["per_run"]
len(per_run)


In [ ]:
# Select one example run (profile) to inspect in detail
run0 = per_run[0]
run0["source_song"], run0["n_songs"]


In [ ]:
final_scores = run0["final_scores"]
cos_sims = run0["cosine_similarities"]
avoid_pens = run0["avoid_penalties"]
fav_overlaps = run0["favorite_overlaps"]

len(final_scores), len(cos_sims), len(avoid_pens), len(fav_overlaps)


In [ ]:
# Scatter: cosine_similarity vs final_score (Pearson check)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(cos_sims, final_scores, alpha=0.7, edgecolor="black")
ax.set_xlabel("cosine_similarity")
ax.set_ylabel("final_score")
ax.set_title("RQ3: final_score vs cosine_similarity (one run)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


For Pearson's *r*, we are looking for an approximately straight-line
relationship between the two variables:

- Points should roughly lie along an increasing line.
- Strong curvature or obvious non-linear patterns would suggest that
  Pearson might not fully capture the relationship.

In this case, because `final_score` is constructed as
`cosine_similarity - 0.5 * avoid_penalty`, a very strong positive
linear relationship between final_score and cosine_similarity is
expected and is desirable as a sanity check.


In [ ]:
# Scatter: avoid_penalty vs final_score (Pearson check)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(avoid_pens, final_scores, alpha=0.7, edgecolor="black")
ax.set_xlabel("avoid_penalty")
ax.set_ylabel("final_score")
ax.set_title("RQ3: final_score vs avoid_penalty (one run)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


Here we expect a **negative** trend: songs with higher avoid_penalty
should tend to have lower final_score. Because the avoid term is a
smaller component of the formula than cosine_similarity, the pattern
will typically be noisier and less "line-like" than in the previous
plot, but a general downward tendency is still expected.


In [ ]:
# Scatter: favorite_overlap vs cosine_similarity (Spearman check)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fav_overlaps, cos_sims, alpha=0.7, edgecolor="black")
ax.set_xlabel("favorite_overlap")
ax.set_ylabel("cosine_similarity")
ax.set_title("RQ3: cosine_similarity vs favorite_overlap (one run)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


Spearman's ρ measures whether there is a **monotonic** relationship
between two variables (as one increases, the other tends to increase or
decrease), without assuming a straight-line shape.

In this scatterplot we are mainly looking for:

- an overall *increasing* trend (songs with higher favourite-note
  overlap also tend to have higher cosine similarity),
- but we do **not** require the points to lie along a straight line.

This justifies the use of Spearman's ρ rather than Pearson's r for this
pair.
